[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C39_Distributed_Training_Course/01_collectives/01_collectives.ipynb)

# 01 · 集合通信原语（用 numpy 在单进程内模拟多 rank）

目标：把 **broadcast / all-reduce / all-gather / reduce-scatter / all-to-all** 的语义、**ring all-reduce 算法**、**α-β 成本模型** 用 numpy 模拟出来，并用 `assert` 与单进程参考**对拍**。

路线：把 rank 模拟成列表 → 五个原语语义 → 核心恒等式 rs+ag==all-reduce → **逐步**模拟 ring all-reduce → 通信量账 → α-β 成本对比 → ✏️ 练习 → 📖 答案 → 🧪 真实集群成本胶囊 → 🔧 torch.distributed 对照。

> 心智模型：**rank = 列表元素；collective = 对列表的纯函数；通信量 = 数出来的字节/步数**。结构与账正确 → 可迁移到 NCCL。

## 1 · 五个原语的语义（先把语义写对）

用长度 `W` 的列表表示 W 个 rank，第 `r` 个元素 = 第 r 个 rank 的张量。先实现 broadcast / reduce / all-reduce 三个最基本的。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def broadcast(shards, src=0):
    '''把 src rank 的张量复制给所有 rank。'''
    return [shards[src].copy() for _ in shards]

def reduce_sum(shards, dst=0):
    '''把所有 rank 求和，只有 dst 拿到结果（其余返回 None）。'''
    total = sum(shards)
    return [total.copy() if r == dst else None for r in range(len(shards))]

def all_reduce_sum(shards):
    '''求和后每个 rank 都拿到同一个和。'''
    total = sum(shards)
    return [total.copy() for _ in shards]

W = 4
shards = [rng.standard_normal(3) for _ in range(W)]
ref = sum(shards)
# broadcast：每个 rank 都等于 src
bc = broadcast(shards, src=0)
assert all(np.allclose(x, shards[0]) for x in bc)
# reduce：只有 dst 有结果
rd = reduce_sum(shards, dst=0)
assert np.allclose(rd[0], ref) and rd[1] is None
# all-reduce：每个 rank 都有结果
ar = all_reduce_sum(shards)
assert all(np.allclose(x, ref) for x in ar)
print('broadcast / reduce / all-reduce 语义全部正确 ✅')
print('all-reduce 后 4 个 rank 的结果一致:', all(np.allclose(ar[0], x) for x in ar))

## 2 · all-gather 与 reduce-scatter：一对对偶

- **all-gather**：每 rank 贡献一片 `shards[r]`，结束后每 rank 都拥有 `concat(所有片)`。
- **reduce-scatter**：把所有 rank 的（完整）张量求和，再把和**切成 W 片**，第 r 个 rank 只拿第 r 片。

注意它们的输入形状不同：all-gather 的输入是「每 rank 一小片」，reduce-scatter 的输入是「每 rank 一个完整张量」。

In [ ]:
def all_gather(shards):
    '''每 rank 贡献 shards[r]，每 rank 都得到 concat。'''
    full = np.concatenate(shards)
    return [full.copy() for _ in shards]

def reduce_scatter_sum(shards):
    '''每 rank 贡献一个完整张量；求和后切 W 片，第 r 个 rank 拿第 r 片。'''
    W = len(shards)
    total = sum(shards)                       # 完整张量的全局和
    assert total.shape[0] % W == 0, '长度需被 W 整除（演示）'
    pieces = np.split(total, W)               # 切成 W 片
    return [pieces[r].copy() for r in range(W)]   # 第 r 个 rank 拿第 r 片

W = 4
# all-gather：每 rank 一小片(长度2)
small = [rng.standard_normal(2) for _ in range(W)]
ag = all_gather(small)
assert ag[0].shape == (8,) and np.allclose(ag[0], np.concatenate(small))
assert all(np.allclose(ag[0], x) for x in ag)
print('all-gather: 4 个长度2的片 -> 每 rank 都拿到长度8的完整张量 ✅')

# reduce-scatter：每 rank 一个完整张量(长度8)
full_each = [rng.standard_normal(8) for _ in range(W)]
rs = reduce_scatter_sum(full_each)
expected_total = sum(full_each)
assert rs[0].shape == (2,)
assert np.allclose(np.concatenate(rs), expected_total)  # 各片拼起来=全局和
print('reduce-scatter: 4 个长度8的张量求和 -> 每 rank 拿长度2的一片(各片拼起来=全局和) ✅')

## 3 · 核心恒等式：reduce-scatter ∘ all-gather == all-reduce

本模块（及模块 02 的 ZeRO）的钥匙：**先 reduce-scatter（每 rank 得到自己那片的全局和），再 all-gather（拼回完整全局和）== all-reduce**。

亲手验证它逐位成立。

In [ ]:
def all_reduce_via_rs_ag(shards):
    '''用 reduce-scatter 接 all-gather 组合出 all-reduce。'''
    rs = reduce_scatter_sum(shards)          # 每 rank 拿到自己那片的全局和
    ag = all_gather(rs)                      # 把各片拼回每 rank 都有的完整全局和
    return ag

W = 4
shards = [rng.standard_normal(8) for _ in range(W)]
via = all_reduce_via_rs_ag(shards)
direct = all_reduce_sum(shards)
ref = sum(shards)
for r in range(W):
    assert np.allclose(via[r], ref), f'rank{r} 组合结果应等于全局和'
    assert np.allclose(via[r], direct[r]), 'rs+ag 应逐位等于直接 all-reduce'
print('✅ 核心恒等式成立：reduce-scatter ∘ all-gather == all-reduce')
print('   这正是 ring all-reduce 的实现骨架，也是 ZeRO/FSDP 的工作原理（模块 02）。')

## 4 · 逐步模拟 ring all-reduce（不是直接 sum，而是真的绕环传 chunk）

把 W 个 rank 排成环，每个张量切成 W 个 chunk。**阶段一 reduce-scatter**（W-1 步）：每步每个 rank 把一个 chunk 发给右邻、把左邻来的累加到自己对应 chunk；W-1 步后 rank_r 持有 chunk_r 的全局和。**阶段二 all-gather**（W-1 步）：把完成的 chunk 绕环传一圈。

我们**逐步**实现，断言最终 == 直接求和——确认这个算法（不只是结果）是对的。

In [ ]:
def ring_all_reduce(shards):
    '''逐步模拟 ring all-reduce。shards[r] 长度需被 W 整除。返回每个 rank 的完整全局和。'''
    W = len(shards)
    n = shards[r0:=0].shape[0]
    assert n % W == 0
    csize = n // W
    # buf[r] 是 rank r 的工作缓冲（切成 W 个 chunk 视图），初始 = 自己的数据
    buf = [shards[r].copy() for r in range(W)]
    def chunk(r, k):                          # rank r 的第 k 个 chunk 的切片范围
        return slice(k * csize, (k + 1) * csize)

    # ---- 阶段一：reduce-scatter（W-1 步）----
    # 第 step 步，rank r 发送 chunk_idx=(r-step) mod W 给右邻，并把左邻发来的累加
    for step in range(W - 1):
        incoming = [None] * W
        for r in range(W):
            send_k = (r - step) % W              # 本步 rank r 要发的 chunk 编号
            right = (r + 1) % W
            incoming[right] = (send_k, buf[r][chunk(r, send_k)].copy())
        for r in range(W):                      # 收下并累加
            k, data = incoming[r]
            buf[r][chunk(r, k)] += data
    # 此时 rank r 的 chunk_(r+1)%W 持有该 chunk 的全局和

    # ---- 阶段二：all-gather（W-1 步）----
    # 把每个 rank 已完成的 chunk 绕环传一圈，覆盖式填满所有 rank 的对应 chunk
    for step in range(W - 1):
        incoming = [None] * W
        for r in range(W):
            send_k = (r + 1 - step) % W          # 本步 rank r 持有的已完成 chunk
            right = (r + 1) % W
            incoming[right] = (send_k, buf[r][chunk(r, send_k)].copy())
        for r in range(W):                      # 覆盖（不是累加）
            k, data = incoming[r]
            buf[r][chunk(r, k)] = data
    return buf

W = 4
shards = [rng.standard_normal(12) for _ in range(W)]   # 12 被 4 整除
out = ring_all_reduce(shards)
ref = sum(shards)
for r in range(W):
    assert np.allclose(out[r], ref, atol=1e-12), f'rank{r} ring 结果应等于全局和'
print('✅ 逐步 ring all-reduce 正确：2×(W-1)=6 步绕环传 chunk 后，每个 rank 都得到全局和')
print('   每一步只传 S/W 字节、只和右邻通信 —— 这正是 NCCL 内部做的事（只是它跨网络并发）。')

## 5 · 通信量的账：ring 每 rank 收发 ≈ 2S，不随 W 爆炸

ring all-reduce 每个 rank 在两阶段各 `W-1` 步、每步发 `S/W` 字节，故每 rank 收发 `2(W-1)/W·S ≈ 2S`。

我们在逐步模拟里**真的数出**每个 rank 发了多少字节，断言它等于公式、且不随 W 爆炸。

In [ ]:
def ring_bytes_sent_per_rank(W, S):
    '''逐步推演 ring，统计单个 rank 发送的总字节(每元素按 bytes_per_elem=1 计 -> 用 S 表示总元素)。'''
    csize = S / W
    sent = 0.0
    for step in range(W - 1):    # 阶段一
        sent += csize
    for step in range(W - 1):    # 阶段二
        sent += csize
    return sent                  # = 2(W-1)*S/W

def naive_total_bytes(W, S):
    '''朴素汇聚-广播的总搬运量：2(W-1)*S（且压在 root 链路）。'''
    return 2 * (W - 1) * S

S = 1_000_000.0
print(f"{'W':>5} {'ring每rank':>12} {'公式2S(W-1)/W':>16} {'朴素总量':>12}")
for W_ in [2, 4, 8, 16, 64, 256]:
    rb = ring_bytes_sent_per_rank(W_, S)
    formula = 2 * (W_ - 1) / W_ * S
    assert abs(rb - formula) < 1e-6
    print(f'{W_:>5} {rb:>12.0f} {formula:>16.0f} {naive_total_bytes(W_, S):>12.0f}')
# ring 每 rank 通信量上界 2S，与 W 无关；朴素总量线性增长
assert ring_bytes_sent_per_rank(100000, S) < 2 * S
print('\n✅ ring 每 rank ≈ 2S 封顶(W→∞)，朴素总量随 W 线性涨 —— 大规模数据并行靠的就是这一点。')

## 6 · α-β 成本模型：大张量用 ring，小张量用 tree

传 n 字节耗时 `T = α + β·n`（α 延迟/每条消息，β=1/带宽）。总耗时 ≈ **步数·α + 字节·β**。

- ring：`2(W-1)` 步、每 rank `2S(W-1)/W` 字节 → `2(W-1)·α + 2S·β`
- tree/递归倍增：`2·log₂W` 步、≈`2S` 字节 → `2log₂W·α + 2S·β`

ring 的 β 项最优但步数(α 项)多；张量小时 α 主导，tree 的 log 步数胜出。

In [ ]:
import math
def cost_ring(W, S, alpha, beta):
    steps = 2 * (W - 1)
    bytes_per_rank = 2 * (W - 1) / W * S
    return steps * alpha + bytes_per_rank * beta

def cost_tree(W, S, alpha, beta):
    steps = 2 * math.ceil(math.log2(W))
    bytes_per_rank = 2 * S            # 近似
    return steps * alpha + bytes_per_rank * beta

alpha = 1e-6        # 1 微秒/消息
beta = 1 / 100e9    # 100 GB/s -> 秒/字节
W = 64
print(f"{'张量大小':>12} {'ring(ms)':>10} {'tree(ms)':>10} {'谁更快':>8}")
for S in [1e3, 1e5, 1e7, 1e9]:
    cr = cost_ring(W, S, alpha, beta) * 1e3
    ct = cost_tree(W, S, alpha, beta) * 1e3
    print(f'{S:>12.0e} {cr:>10.4f} {ct:>10.4f} {"ring" if cr < ct else "tree":>8}')
# 小张量 tree 快(步数少)，大张量 ring 快(带宽优)
assert cost_tree(W, 1e3, alpha, beta) < cost_ring(W, 1e3, alpha, beta), '小张量 tree 应更快'
assert cost_ring(W, 1e9, alpha, beta) < cost_tree(W, 1e9, alpha, beta), '大张量 ring 应更快'
print('\n✅ 验证取舍：小张量 tree(延迟最优)胜，大张量 ring(带宽最优)胜 —— NCCL 据此自动切换。')

---
## ✏️ 练习 1：all-reduce 求平均（数据并行同步梯度的形式）

数据并行同步梯度用的是 **all-reduce 求平均**：`(1/W) Σ g_r`。

实现 `all_reduce_mean(shards)`：每个 rank 返回所有 rank 的**平均**（不是和）。

In [ ]:
def all_reduce_mean(shards):
    # TODO: 求和后除以 W，每个 rank 返回同一个平均值
    #       提示：可复用 all_reduce_sum 再 /W
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
W = 5
shards = [rng.standard_normal(4) for _ in range(W)]
out = all_reduce_mean(shards)
ref = sum(shards) / W
assert all(np.allclose(x, ref) for x in out), '每个 rank 应得到平均值'
assert all(np.allclose(out[0], x) for x in out), '所有 rank 结果应一致'
print('✅ 练习 1 通过：all-reduce 求平均 == 梯度同步的形式')

## ✏️ 练习 2：ring reduce-scatter 的分步正确性

只实现 ring all-reduce 的**第一阶段**（reduce-scatter），断言 W-1 步后 rank_r 的第 `(r+1)%W` 个 chunk 持有该 chunk 的全局和。

实现 `ring_reduce_scatter(shards)`：返回每个 rank 的 buffer，使得 `buf[r]` 在 chunk `(r+1)%W` 处 == 全局和的对应片。

In [ ]:
def ring_reduce_scatter(shards):
    W = len(shards)
    n = shards[0].shape[0]
    csize = n // W
    buf = [shards[r].copy() for r in range(W)]
    def chunk(r, k):
        return slice(k * csize, (k + 1) * csize)
    # TODO: W-1 步，每步 rank r 发 chunk (r-step)%W 给右邻并累加左邻来的
    #       （照搬第 4 节阶段一的逻辑）
    raise NotImplementedError
    return buf

In [ ]:
# —— 练习 2 自测 ——
W = 4
shards = [rng.standard_normal(8) for _ in range(W)]
buf = ring_reduce_scatter(shards)
total = sum(shards)
csize = 8 // W
for r in range(W):
    k = (r + 1) % W                      # rank r 完成的 chunk 编号
    got = buf[r][k * csize:(k + 1) * csize]
    want = total[k * csize:(k + 1) * csize]
    assert np.allclose(got, want), f'rank{r} 的 chunk{k} 应是全局和'
print('✅ 练习 2 通过：ring 阶段一(reduce-scatter)后，每个 rank 各持有一片完成的全局和')

## ✏️ 练习 3：reduce-scatter + all-gather ≡ all-reduce（自己拼一遍）

用你的 `reduce_scatter_sum` 和 `all_gather`（第 2 节已给）组合出 all-reduce，并对拍直接 all-reduce。

实现 `my_all_reduce(shards)`，断言它逐位等于 `sum(shards)`。

In [ ]:
def my_all_reduce(shards):
    # TODO: 先 reduce_scatter_sum，再 all_gather；返回每个 rank 的完整全局和
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
W = 4
shards = [rng.standard_normal(12) for _ in range(W)]
out = my_all_reduce(shards)
ref = sum(shards)
assert all(np.allclose(x, ref) for x in out), '组合结果应等于全局和'
print('✅ 练习 3 通过：亲手用两个对偶原语拼出 all-reduce')

## ✏️ 练习 4：通信量公式

实现 `comm_volume(kind, W, S)` 返回**每个 rank** 收发的字节数：
- `'ring_allreduce'` → `2(W-1)/W · S`
- `'all_gather'` → `(W-1)/W · S`（每 rank 收 W-1 片、每片 S/W）
- `'reduce_scatter'` → `(W-1)/W · S`

并验证 `ring_allreduce == all_gather + reduce_scatter`（呼应核心恒等式的成本版）。

In [ ]:
def comm_volume(kind, W, S):
    # TODO: 按上面三条公式返回每 rank 字节数
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
W, S = 8, 1_000_000.0
ar = comm_volume('ring_allreduce', W, S)
ag = comm_volume('all_gather', W, S)
rs = comm_volume('reduce_scatter', W, S)
assert abs(ar - 2 * (W - 1) / W * S) < 1e-6
assert abs(ag - (W - 1) / W * S) < 1e-6
assert abs(ar - (ag + rs)) < 1e-6, 'all-reduce 成本 == all-gather + reduce-scatter'
print(f'ring all-reduce/rank = {ar:.0f}B = all-gather({ag:.0f}) + reduce-scatter({rs:.0f})')
print('✅ 练习 4 通过：通信量公式与恒等式的成本版都对上了')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def all_reduce_mean(shards):
    summed = all_reduce_sum(shards)
    return [x / len(shards) for x in summed]

In [ ]:
# 练习 2 参考答案
def ring_reduce_scatter(shards):
    W = len(shards)
    n = shards[0].shape[0]
    csize = n // W
    buf = [shards[r].copy() for r in range(W)]
    def chunk(r, k):
        return slice(k * csize, (k + 1) * csize)
    for step in range(W - 1):
        incoming = [None] * W
        for r in range(W):
            send_k = (r - step) % W
            right = (r + 1) % W
            incoming[right] = (send_k, buf[r][chunk(r, send_k)].copy())
        for r in range(W):
            k, data = incoming[r]
            buf[r][chunk(r, k)] += data
    return buf

In [ ]:
# 练习 3 参考答案
def my_all_reduce(shards):
    rs = reduce_scatter_sum(shards)
    return all_gather(rs)

In [ ]:
# 练习 4 参考答案
def comm_volume(kind, W, S):
    if kind == 'ring_allreduce':
        return 2 * (W - 1) / W * S
    elif kind in ('all_gather', 'reduce_scatter'):
        return (W - 1) / W * S
    else:
        raise ValueError(kind)

---
## 🧪 真实数据胶囊：算一次真实梯度 all-reduce 要多久

用**真实**模型与集群参数，把本模块的成本模型接到现实：一个 7B 模型的梯度（fp16，2 字节/参数）做一次 ring all-reduce，在 InfiniBand（约 100 GB/s 等效总线带宽）上要多久？这决定了数据并行每步的通信开销。

In [ ]:
# 真实参数（公开约数）；try 在线取参数量，except 回退到内置真实值
def get_model_params(name='Llama-2-7B'):
    try:
        # 真实环境可从 HF config 读 num_parameters；离线回退到公开值
        raise RuntimeError('offline')
    except Exception:
        return {'Llama-2-7B': 6.7e9, 'Llama-2-70B': 6.9e10}[name]

P = get_model_params('Llama-2-7B')
bytes_per_param = 2                 # fp16 梯度
S = P * bytes_per_param            # 梯度张量总字节
W = 64                             # 64 路数据并行
bus_bw = 100e9                     # 100 GB/s 等效
alpha = 5e-6                       # 5 us/消息
beta = 1 / bus_bw

bytes_per_rank = 2 * (W - 1) / W * S
t = 2 * (W - 1) * alpha + bytes_per_rank * beta
print(f'7B 梯度 = {S/1e9:.1f} GB (fp16)')
print(f'ring all-reduce 每 rank 收发 ≈ {bytes_per_rank/1e9:.2f} GB')
print(f'估计单次 all-reduce 耗时 ≈ {t*1e3:.1f} ms')
assert bytes_per_rank < 2 * S
print('\n这就是为什么要把通信和反向计算重叠(模块 02 DDP)：否则每步白等这么久。')

**🧪 胶囊练习**：实现 `allreduce_ms(P, bytes_per_param, W, bus_bw_gbps, alpha_us)`，返回一次 ring all-reduce 的毫秒数。用它算 70B 模型在 256 路数据并行、200 GB/s 下的耗时。

In [ ]:
def allreduce_ms(P, bytes_per_param, W, bus_bw_gbps, alpha_us):
    # TODO: S=P*bytes_per_param; beta=1/(bus_bw_gbps*1e9); alpha=alpha_us*1e-6
    #       t = 2(W-1)*alpha + 2(W-1)/W*S*beta; 返回 t*1e3
    raise NotImplementedError

In [ ]:
# 自测
ms = allreduce_ms(6.9e10, 2, 256, 200, 5)
assert ms > 0
# 与手算对照
S = 6.9e10 * 2; beta = 1/200e9; alpha = 5e-6
expect = (2*(256-1)*alpha + 2*(256-1)/256*S*beta) * 1e3
assert abs(ms - expect) < 1e-6
print(f'70B 梯度 256 路数据并行 @200GB/s: 单次 all-reduce ≈ {ms:.0f} ms')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def allreduce_ms(P, bytes_per_param, W, bus_bw_gbps, alpha_us):
    S = P * bytes_per_param
    beta = 1 / (bus_bw_gbps * 1e9)
    alpha = alpha_us * 1e-6
    t = 2 * (W - 1) * alpha + 2 * (W - 1) / W * S * beta
    return t * 1e3

---
## 🔧 旁注：对应的 torch.distributed 长什么样

本模块模拟的每个原语，在真实多 GPU 上就是一行 `torch.distributed` 调用（伪代码，**本环境不跑**）：

```python
import torch, torch.distributed as dist
dist.init_process_group(backend='nccl')   # ring/tree 算法由 NCCL 自动选

dist.all_reduce(t, op=dist.ReduceOp.SUM)   # == 我们的 all_reduce_sum
dist.all_gather(out_list, t)               # == 我们的 all_gather
dist.reduce_scatter(out, in_list)          # == 我们的 reduce_scatter_sum
dist.broadcast(t, src=0)                   # == 我们的 broadcast
dist.all_to_all(out_list, in_list)         # == all-to-all（MoE 路由）
dist.barrier()                             # 同步屏障
```

对应关系一一对得上：语义相同、通信量相同。**差别只在**：NCCL 跨网络并发执行、自动用 ring/tree、按拓扑优化——这些都是「第二步」的工程，本课负责的「第一步」（语义对、账对）你已经亲手验证过了。

### 小结
- 分布式通信 = 几个**集合原语**的组合；每个原语有确定的**语义**和确定的**成本**。
- 核心恒等式：**all-reduce = reduce-scatter ∘ all-gather**（ring 的骨架，也是 ZeRO/FSDP 的原理）。
- **ring all-reduce**：每 rank 收发 ≈ 2S，**不随 W 爆炸** → 数据并行能扩到万卡。
- **α-β 模型**：大张量用 ring（带宽最优）、小张量用 tree（延迟最优）；跨节点带宽更低 → 决定 3D 并行布局。

下一站：**模块 02 · 数据并行与 FSDP** —— 用 all-reduce/reduce-scatter/all-gather 把模型铺到多卡、把冗余显存挤掉。